# 5.6. GPU

在本节中，我们将讨论如何利用这种计算性能进行研究。首先是如何使用单个GPU，然后是如何使用多个GPU和多个服务器（具有多个GPU）。

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

print(tf.__version__)

## 5.6.1. 计算设备

In [ ]:
# 查询可用gpu数量
print(len(tf.config.list_physical_devices('GPU')))

In [ ]:
# 查询所有可用设备
print(tf.config.list_physical_devices())

In [ ]:
# 定义两个便利的函数
def try_gpu(i=0):
    """如果存在，则返回gpu(i)，否则返回cpu()"""
    if len(tf.config.list_physical_devices('GPU')) >= i + 1:
        return tf.device(f'/GPU:{i}')
    return tf.device('/CPU:0')

def try_all_gpus():
    """返回所有可用的GPU，如果没有GPU，则返回[cpu()]"""
    num_gpus = len(tf.config.list_physical_devices('GPU'))
    devices = [tf.device(f'/GPU:{i}') for i in range(num_gpus)]
    return devices if devices else [tf.device('/CPU:0')]

try_gpu(), try_gpu(10), try_all_gpus()

## 5.6.2. 张量与GPU

In [ ]:
# 默认情况下，张量是在CPU上创建的
x = tf.constant([1, 2, 3])
print(x.device)

### 5.6.2.1. 存储在GPU上

In [ ]:
# 在第一个GPU上创建张量
with try_gpu():
    X = tf.ones((2, 3))
print(X)

### 5.6.2.2. 复制

In [ ]:
# 假设有至少两个GPU，下面的代码将在第二个GPU上创建一个随机张量
with try_gpu(1):
    Y = tf.random.uniform((2, 3))
print(Y)

In [ ]:
# 要计算X + Y，我们需要决定在哪里执行这个操作
with try_gpu():
    Z = X
print(X)
print(Z)

### 5.6.2.3. 旁注

In [ ]:
# 如果张量已经在目标设备上，操作就不需要复制
with try_gpu():
    Z2 = Z
print(Z2 is Z)

## 5.6.3. 神经网络与GPU

In [ ]:
# 神经网络模型在GPU上
strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    net = tf.keras.models.Sequential([
        tf.keras.layers.Dense(1)
    ])

In [ ]:
# 当输入为GPU上的张量时，模型将在同一GPU上计算结果
with try_gpu():
    net(X)

In [ ]:
# 确认模型参数存储在同一个GPU上
print(net.layers[0].weights[0].device)

## 小结

- 我们可以指定用于存储和计算的设备，例如CPU或GPU。默认情况下，数据在主内存中创建，然后使用CPU进行计算。
- 深度学习框架要求计算的所有输入数据都在同一设备上，无论是CPU还是GPU。
- 不经意地移动数据可能会显著降低性能。一个典型的错误如下：计算GPU上每个小批量的损失，并在命令行中将其报告给用户（或将其记录在NumPy ndarray中）时，将触发全局解释器锁，从而使所有GPU阻塞。最好是为GPU内部的日志分配内存，并且只移动较大的日志。